In [2]:
import os
import subprocess
import json
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

from pykrx import stock
import pandas as pd

MAX_WORKERS = 3
REFRESH_EVERY_N = 150
REQUEST_DELAY = 0.3

_refresh_lock = threading.Lock()
_api_call_count = 0

KRX 로그인 시도...
  로그인 ID: hard8903
KRX 로그인 완료.
  로그인 시간: 2026-07-18 22:02:07
  만료 시간: 2026-07-18 23:02:07


In [3]:
def refresh_token():
    result = subprocess.run(["kiwoomcli", "auth", "refresh"], capture_output=True, text=True, encoding="utf-8")
    return result.returncode == 0

def get_all_tickers():
    today = datetime.today().strftime("%Y%m%d")
    kospi = stock.get_market_ticker_list(today, market="KOSPI")
    kosdaq = stock.get_market_ticker_list(today, market="KOSDAQ")
    all_tickers = [(t, "KOSPI") for t in kospi] + [(t, "KOSDAQ") for t in kosdaq]
    valid = [(t, m) for t, m in all_tickers if t.isdigit()]
    print(f"전체 {len(all_tickers)}개 중 유효 코드 {len(valid)}개")
    return valid

def parse_price(s):
    if s in (None, ""):
        return None
    return abs(int(s))

def fetch_candles(ticker_market, interval, data_dir, retry=True):
    global _api_call_count
    ticker, market = ticker_market
    filepath = os.path.join(data_dir, f"{ticker}.csv")
    if os.path.exists(filepath):
        return (ticker, "skip")

    result = subprocess.run(
        ["kiwoomcli", "domestic", "candles", "stock-minute",
         "--code", ticker, "--interval", interval, "--pages", "0", "--format", "json"],
        capture_output=True, text=True, encoding="utf-8"
    )

    if result.returncode != 0:
        err = result.stderr or ""
        if retry and any(k in err for k in ["토큰","인증","expired","unauthorized","401"]):
            with _refresh_lock:
                refresh_token()
            return fetch_candles(ticker_market, interval, data_dir, retry=False)
        return (ticker, f"error: {err[:200]}")

    try:
        data = json.loads(result.stdout, strict=False)
    except Exception as e:
        return (ticker, f"parse_error: {e}")

    records = data.get("stk_min_pole_chart_qry")
    if not records:
        return (ticker, "empty")

    df = pd.DataFrame(records)
    for col in ["cur_prc","open_pric","high_pric","low_pric"]:
        df[col] = df[col].apply(parse_price)
    df["cntr_tm"] = pd.to_datetime(df["cntr_tm"], format="%Y%m%d%H%M%S", errors="coerce")
    df = df.dropna(subset=["cntr_tm"]).sort_values("cntr_tm").reset_index(drop=True)
    df.to_csv(filepath, index=False, encoding="utf-8-sig")

    time.sleep(REQUEST_DELAY)
    with _refresh_lock:
        _api_call_count += 1
        if _api_call_count % REFRESH_EVERY_N == 0:
            refresh_token()

    return (ticker, "ok")

def run_collection(interval, data_dir):
    os.makedirs(data_dir, exist_ok=True)
    tickers = get_all_tickers()
    total = len(tickers)

    ok, skip, fail = 0, 0, 0
    failed_list = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_candles, tm, interval, data_dir): tm for tm in tickers}
        with tqdm(total=total, desc=f"{interval}분봉 수집") as pbar:
            for future in as_completed(futures):
                ticker, result = future.result()
                if result == "ok":
                    ok += 1
                elif result == "skip":
                    skip += 1
                else:
                    fail += 1
                    failed_list.append((ticker, result))
                pbar.set_postfix(성공=ok, 스킵=skip, 실패=fail)
                pbar.update(1)

    print(f"\n완료: 성공 {ok} / 스킵 {skip} / 실패 {fail}")
    if failed_list:
        print("실패 샘플:", failed_list[:10])

In [ ]:
run_collection(interval="15", data_dir="min15_data")

전체 2765개 중 유효 코드 2690개


15분봉 수집:   3%|▎         | 72/2690 [01:16<46:15,  1.06s/it, 성공=0, 스킵=0, 실패=72]  


In [1]:
import subprocess
result = subprocess.run(
    ["kiwoomcli", "domestic", "candles", "stock-minute",
     "--code", "005930", "--interval", "15", "--pages", "0", "--format", "json"],
    capture_output=True, text=True, encoding="utf-8"
)
print("returncode:", result.returncode)
print("stdout:", result.stdout[:500])
print("stderr:", result.stderr[:500])

returncode: 0
stdout: {"return_msg":"인증에 실패했습니다[8005:Token이 유효하지 않습니다]","return_code":3}

stderr: 


In [2]:
import subprocess
result = subprocess.run(
    ["kiwoomcli", "domestic", "candles", "stock-minute",
     "--code", "005930", "--interval", "15", "--pages", "0", "--format", "json"],
    capture_output=True, text=True, encoding="utf-8"
)
print("returncode:", result.returncode)
print("stdout:", result.stdout[:500])

returncode: 0
stdout: {"stk_cd":"005930","stk_min_pole_chart_qry":[{"cur_prc":"-255000","trde_qty":"3946529","cntr_tm":"20260716153000","open_pric":"-255000","high_pric":"-255000","low_pric":"-255000","acc_trde_qty":"26605890","pred_pre":"-24500","pred_pre_sig":"5"},{"cur_prc":"-256500","trde_qty":"391027","cntr_tm":"20260716151500","open_pric":"-254500","high_pric":"-256500","low_pric":"-254000","acc_trde_qty":"22659361","pred_pre":"-23000","pred_pre_sig":"5"},{"cur_prc":"-254500","trde_qty":"2146005","cntr_tm":"202


In [4]:
def fetch_candles(ticker_market, interval, data_dir, retry=True):
    global _api_call_count
    ticker, market = ticker_market
    filepath = os.path.join(data_dir, f"{ticker}.csv")
    if os.path.exists(filepath):
        return (ticker, "skip")

    result = subprocess.run(
        ["kiwoomcli", "domestic", "candles", "stock-minute",
         "--code", ticker, "--interval", interval, "--pages", "0", "--format", "json"],
        capture_output=True, text=True, encoding="utf-8"
    )

    # stdout 안에 담긴 인증 에러도 감지하도록 보강
    auth_error_in_body = False
    if result.returncode == 0 and result.stdout:
        try:
            preview = json.loads(result.stdout, strict=False)
            if preview.get("return_code") != 0:
                auth_error_in_body = True
        except Exception:
            pass

    if result.returncode != 0 or auth_error_in_body:
        err = result.stderr or result.stdout or ""
        if retry and any(k in err for k in ["토큰","인증","expired","unauthorized","401","유효하지"]):
            with _refresh_lock:
                refresh_token()
            return fetch_candles(ticker_market, interval, data_dir, retry=False)
        return (ticker, f"error: {err[:200]}")

    try:
        data = json.loads(result.stdout, strict=False)
    except Exception as e:
        return (ticker, f"parse_error: {e}")

    records = data.get("stk_min_pole_chart_qry")
    if not records:
        return (ticker, "empty")

    df = pd.DataFrame(records)
    for col in ["cur_prc","open_pric","high_pric","low_pric"]:
        df[col] = df[col].apply(parse_price)
    df["cntr_tm"] = pd.to_datetime(df["cntr_tm"], format="%Y%m%d%H%M%S", errors="coerce")
    df = df.dropna(subset=["cntr_tm"]).sort_values("cntr_tm").reset_index(drop=True)
    df.to_csv(filepath, index=False, encoding="utf-8-sig")

    time.sleep(REQUEST_DELAY)
    with _refresh_lock:
        _api_call_count += 1
        if _api_call_count % REFRESH_EVERY_N == 0:
            refresh_token()

    return (ticker, "ok")

## 다시


In [1]:
import os
import subprocess
import json
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

from pykrx import stock
import pandas as pd

MAX_WORKERS = 3
REFRESH_EVERY_N = 150
REQUEST_DELAY = 0.3

_refresh_lock = threading.Lock()
_api_call_count = 0

KRX 로그인 시도...
  로그인 ID: hard8903
KRX 로그인 완료.
  로그인 시간: 2026-07-18 22:12:25
  만료 시간: 2026-07-18 23:12:25


In [2]:
def refresh_token():
    result = subprocess.run(["kiwoomcli", "auth", "refresh"], capture_output=True, text=True, encoding="utf-8")
    return result.returncode == 0

def get_all_tickers():
    today = datetime.today().strftime("%Y%m%d")
    kospi = stock.get_market_ticker_list(today, market="KOSPI")
    kosdaq = stock.get_market_ticker_list(today, market="KOSDAQ")
    all_tickers = [(t, "KOSPI") for t in kospi] + [(t, "KOSDAQ") for t in kosdaq]
    valid = [(t, m) for t, m in all_tickers if t.isdigit()]
    print(f"전체 {len(all_tickers)}개 중 유효 코드 {len(valid)}개")
    return valid

def parse_price(s):
    if s in (None, ""):
        return None
    return abs(int(s))

def fetch_candles(ticker_market, interval, data_dir, retry=True):
    global _api_call_count
    ticker, market = ticker_market
    filepath = os.path.join(data_dir, f"{ticker}.csv")
    if os.path.exists(filepath):
        return (ticker, "skip")

    result = subprocess.run(
        ["kiwoomcli", "domestic", "candles", "stock-minute",
         "--code", ticker, "--interval", interval, "--pages", "0", "--format", "json"],
        capture_output=True, text=True, encoding="utf-8"
    )

    auth_error_in_body = False
    if result.returncode == 0 and result.stdout:
        try:
            preview = json.loads(result.stdout, strict=False)
            if preview.get("return_code") != 0:
                auth_error_in_body = True
        except Exception:
            pass

    if result.returncode != 0 or auth_error_in_body:
        err = result.stderr or result.stdout or ""
        if retry and any(k in err for k in ["토큰","인증","expired","unauthorized","401","유효하지"]):
            with _refresh_lock:
                refresh_token()
            return fetch_candles(ticker_market, interval, data_dir, retry=False)
        return (ticker, f"error: {err[:200]}")

    try:
        data = json.loads(result.stdout, strict=False)
    except Exception as e:
        return (ticker, f"parse_error: {e}")

    records = data.get("stk_min_pole_chart_qry")
    if not records:
        return (ticker, "empty")

    df = pd.DataFrame(records)
    for col in ["cur_prc","open_pric","high_pric","low_pric"]:
        df[col] = df[col].apply(parse_price)
    df["cntr_tm"] = pd.to_datetime(df["cntr_tm"], format="%Y%m%d%H%M%S", errors="coerce")
    df = df.dropna(subset=["cntr_tm"]).sort_values("cntr_tm").reset_index(drop=True)
    df.to_csv(filepath, index=False, encoding="utf-8-sig")

    time.sleep(REQUEST_DELAY)
    with _refresh_lock:
        _api_call_count += 1
        if _api_call_count % REFRESH_EVERY_N == 0:
            refresh_token()

    return (ticker, "ok")

def run_collection(interval, data_dir):
    os.makedirs(data_dir, exist_ok=True)
    tickers = get_all_tickers()
    total = len(tickers)

    ok, skip, fail = 0, 0, 0
    failed_list = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_candles, tm, interval, data_dir): tm for tm in tickers}
        with tqdm(total=total, desc=f"{interval}분봉 수집") as pbar:
            for future in as_completed(futures):
                ticker, result = future.result()
                if result == "ok":
                    ok += 1
                elif result == "skip":
                    skip += 1
                else:
                    fail += 1
                    failed_list.append((ticker, result))
                pbar.set_postfix(성공=ok, 스킵=skip, 실패=fail)
                pbar.update(1)

    print(f"\n완료: 성공 {ok} / 스킵 {skip} / 실패 {fail}")
    if failed_list:
        print("실패 샘플:", failed_list[:10])

In [3]:
run_collection(interval="15", data_dir="min15_data")

전체 2765개 중 유효 코드 2690개


15분봉 수집: 100%|██████████| 2690/2690 [1:00:19<00:00,  1.35s/it, 성공=1757, 스킵=96, 실패=837]


완료: 성공 1757 / 스킵 96 / 실패 837
실패 샘플: [('138930', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('006840', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('097955', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('000480', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('005830', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('097230', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('016610', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 API 요청 개수를 초과하였습니다. 유량=5, API ID=ka10080] (return_code=5)\n'), ('003580', 'error: HTTP 요청 실패 (429): 허용된 요청 개수를 초과하였습니다[1700:허용된 A

In [5]:
import os
import subprocess
import json
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

from pykrx import stock
import pandas as pd

MAX_WORKERS = 1        # 429 방지를 위해 1로 낮춤
REFRESH_EVERY_N = 150
REQUEST_DELAY = 0.5    # 429 방지를 위해 늘림

_refresh_lock = threading.Lock()
_api_call_count = 0

In [6]:
def refresh_token():
    result = subprocess.run(["kiwoomcli", "auth", "refresh"], capture_output=True, text=True, encoding="utf-8")
    return result.returncode == 0

def get_all_tickers():
    today = datetime.today().strftime("%Y%m%d")
    kospi = stock.get_market_ticker_list(today, market="KOSPI")
    kosdaq = stock.get_market_ticker_list(today, market="KOSDAQ")
    all_tickers = [(t, "KOSPI") for t in kospi] + [(t, "KOSDAQ") for t in kosdaq]
    valid = [(t, m) for t, m in all_tickers if t.isdigit()]
    print(f"전체 {len(all_tickers)}개 중 유효 코드 {len(valid)}개")
    return valid

def parse_price(s):
    if s in (None, ""):
        return None
    return abs(int(s))

def fetch_candles(ticker_market, interval, data_dir, retry=True):
    global _api_call_count
    ticker, market = ticker_market
    filepath = os.path.join(data_dir, f"{ticker}.csv")
    if os.path.exists(filepath):
        return (ticker, "skip")

    result = subprocess.run(
        ["kiwoomcli", "domestic", "candles", "stock-minute",
         "--code", ticker, "--interval", interval, "--pages", "0", "--format", "json"],
        capture_output=True, text=True, encoding="utf-8"
    )

    auth_error_in_body = False
    if result.returncode == 0 and result.stdout:
        try:
            preview = json.loads(result.stdout, strict=False)
            if preview.get("return_code") != 0:
                auth_error_in_body = True
        except Exception:
            pass

    if result.returncode != 0 or auth_error_in_body:
        err = result.stderr or result.stdout or ""
        if "429" in err or "허용된" in err:
            time.sleep(2.0)
            if retry:
                return fetch_candles(ticker_market, interval, data_dir, retry=False)
            return (ticker, f"error: {err[:200]}")
        if retry and any(k in err for k in ["토큰","인증","expired","unauthorized","401","유효하지"]):
            with _refresh_lock:
                refresh_token()
            return fetch_candles(ticker_market, interval, data_dir, retry=False)
        return (ticker, f"error: {err[:200]}")

    try:
        data = json.loads(result.stdout, strict=False)
    except Exception as e:
        return (ticker, f"parse_error: {e}")

    records = data.get("stk_min_pole_chart_qry")
    if not records:
        return (ticker, "empty")

    df = pd.DataFrame(records)
    for col in ["cur_prc","open_pric","high_pric","low_pric"]:
        df[col] = df[col].apply(parse_price)
    df["cntr_tm"] = pd.to_datetime(df["cntr_tm"], format="%Y%m%d%H%M%S", errors="coerce")
    df = df.dropna(subset=["cntr_tm"]).sort_values("cntr_tm").reset_index(drop=True)
    df.to_csv(filepath, index=False, encoding="utf-8-sig")

    time.sleep(REQUEST_DELAY)
    with _refresh_lock:
        _api_call_count += 1
        if _api_call_count % REFRESH_EVERY_N == 0:
            refresh_token()

    return (ticker, "ok")

def run_collection(interval, data_dir):
    os.makedirs(data_dir, exist_ok=True)
    tickers = get_all_tickers()
    total = len(tickers)

    ok, skip, fail = 0, 0, 0
    failed_list = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_candles, tm, interval, data_dir): tm for tm in tickers}
        with tqdm(total=total, desc=f"{interval}분봉 수집") as pbar:
            for future in as_completed(futures):
                ticker, result = future.result()
                if result == "ok":
                    ok += 1
                elif result == "skip":
                    skip += 1
                else:
                    fail += 1
                    failed_list.append((ticker, result))
                pbar.set_postfix(성공=ok, 스킵=skip, 실패=fail)
                pbar.update(1)

    print(f"\n완료: 성공 {ok} / 스킵 {skip} / 실패 {fail}")
    if failed_list:
        print("실패 샘플:", failed_list[:10])

In [ ]:
run_collection(interval="15", data_dir="min15_data")

KRX 세션 만료, 재로그인 시도...
KRX 세션 갱신 완료.
전체 2765개 중 유효 코드 2690개


15분봉 수집:  83%|████████▎ | 2238/2690 [55:43<17:14,  2.29s/it, 성공=729, 스킵=1510, 실패=0]  